In [3]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import os

**For Pushing Image into HF**

In [9]:
from datasets import Dataset, DatasetDict, Image as HFImage, Features, Value, ClassLabel
from huggingface_hub import login
import os

DATA_ROOT = "/kaggle/input/datasets/benjaminkz/places365"
DATASET_NAME = "Punnarunwuwu/industry-verification-seml"

# ================= YOUR MAPPING =================
MAPPING = {
    'beauty_salon':0, 'barbershop':0,
    'drugstore':1, 'pharmacy':1, 'hospital_room':1,
    'restaurant':2, 'coffee_shop':2, 'fast_food_restaurant':2, 'bar':2, 'cafeteria':2, 'food_court':2, 'sushi_bar':2, 'pizzeria':2, 'ice_cream_parlor':2, 'bakery/shop':2, 'diner':2,
    'movie_theater/indoor':3, 'cinema':3, 'bowling_alley':3,
    'apartment_building/outdoor':4, 'living_room':4, 'bedroom':4, 'house':4,
    'supermarket':5, 'department_store':5, 'clothing_store':5, 'shoe_shop':5, 'bookstore':5, 'grocery_store':5
}

# ================= FEATURES =================
features = Features({
    "image": HFImage(),
    "label": ClassLabel(names=[str(i) for i in range(6)]),
    "class_name": Value("string")
})

# ================= GENERATOR FUNCTION =================
def image_generator(split_path, mapping):
    """Yields images one by one directly into the Hugging Face dataset."""
    for class_name, label in mapping.items():
        path_nested = os.path.join(split_path, class_name)
        path_underscore = os.path.join(split_path, class_name.replace("/", "_"))
        
        target_path = None
        if os.path.isdir(path_nested):
            target_path = path_nested
        elif os.path.isdir(path_underscore):
            target_path = path_underscore
            
        if not target_path:
            continue
            
        for file in os.listdir(target_path):
            if not file.lower().endswith(('.png', '.jpg', '.jpeg')):
                continue
                
            image_path = os.path.join(target_path, file)
            yield {
                "image": {"path": image_path},
                "label": label,
                "class_name": class_name
            }

# ================= BUILD DATASET DICT =================
dataset_dict = {}

for split in ["train", "val"]:
    split_path = os.path.join(DATA_ROOT, split)
    
    if not os.path.isdir(split_path):
        print(f"⚠️ Skipping '{split}', folder not found.")
        continue
        
    print(f"Generating Hugging Face dataset for '{split}'... (Look for the built-in progress bar!)")
    
    # This is the fast, memory-safe way!
    dataset = Dataset.from_generator(
        image_generator, 
        gen_kwargs={"split_path": split_path, "mapping": MAPPING},
        features=features
    )
    
    print("Shuffling...")
    dataset = dataset.shuffle(seed=42)
    
    hf_split_name = "validation" if split == "val" else split
    dataset_dict[hf_split_name] = dataset
    print(f"'{split}' dataset built successfully!\n")

# ================= PUSH =================
if not dataset_dict:
    raise ValueError("No images were found in either split. Check your MAPPING keys!")

hf_dataset = DatasetDict(dataset_dict)

print("Pushing to Hugging Face Hub (this will upload optimized Parquet chunks)...")
hf_dataset.push_to_hub(
    DATASET_NAME,
    private=False,
    embed_external_files=True,
    max_shard_size="500MB"
)

print("✅ Done!")

Generating Hugging Face dataset for 'train'... (Look for the built-in progress bar!)


Generating train split: 0 examples [00:00, ? examples/s]

Shuffling...
'train' dataset built successfully!

Generating Hugging Face dataset for 'val'... (Look for the built-in progress bar!)


Generating train split: 0 examples [00:00, ? examples/s]

Shuffling...
'val' dataset built successfully!

Pushing to Hugging Face Hub (this will upload optimized Parquet chunks)...


...

Map:   0%|          | 0/2100 [00:00<?, ? examples/s]

Creating parquet from Arrow format:   0%|          | 0/21 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

✅ Done!


**Download DataSet from HuggingFace**

In [12]:
from datasets import load_dataset

dataset = load_dataset("Punnarunwuwu/industry-verification-seml")

README.md:   0%|          | 0.00/631 [00:00<?, ?B/s]

data/train-00000-of-00003.parquet:   0%|          | 0.00/490M [00:00<?, ?B/s]

data/train-00001-of-00003.parquet:   0%|          | 0.00/490M [00:00<?, ?B/s]

data/train-00002-of-00003.parquet:   0%|          | 0.00/489M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/29.5M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/103781 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2100 [00:00<?, ? examples/s]

In [10]:
def prepare_split(root,mapping,train_n=1800,test_n=200):
    train_data=[]
    test_data=[]
    target_to_sources={i:[] for i in range(6)}

    for src,tgt in mapping.items():
        target_to_sources[tgt].append(src)

    print("\nPreparing dataset split...")
    for tgt,sources in target_to_sources.items():
        paths=[]
        for s in sources:
            p=os.path.join(root,s)
            if os.path.exists(p):
                paths+=glob.glob(os.path.join(p,"*.jpg"))

        random.shuffle(paths)
        selected=paths[:train_n+test_n]

        train_paths=selected[:train_n]
        test_paths=selected[train_n:train_n+test_n]

        train_data += [(p,tgt) for p in train_paths]
        test_data  += [(p,tgt) for p in test_paths]

        print(f"Class {tgt} -> train {len(train_paths)} | test {len(test_paths)}")

    return train_data,test_data

In [13]:
# =========================
# PLACES365 CLASSIFICATION PIPELINE (Hugging Face Edition)
# RESNET34 + PCA + LOGISTIC REGRESSION
# =========================

import random
import numpy as np
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.models import resnet34, ResNet34_Weights

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.decomposition import PCA

# --- NEW HUGGING FACE IMPORT ---
from datasets import load_dataset 

# reproducibility
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# ================= DATASET =================
print("\nDownloading/Loading dataset from Hugging Face...")
dataset = load_dataset("Punnarunwuwu/industry-verification-seml")

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Hugging Face on-the-fly transform
def apply_transforms(batch):
    pixel_values = []
    for img in batch["image"]:
        try:
            # Convert to RGB to avoid channels crashing on grayscale
            pixel_values.append(transform(img.convert("RGB")))
        except Exception:
            # Fallback for any corrupted images
            pixel_values.append(torch.zeros(3, 224, 224))
            
    batch["pixel_values"] = pixel_values
    return batch

# Apply the transform to the dataset
dataset.set_transform(apply_transforms)

# Collate function to format the Hugging Face batch for PyTorch
def collate_fn(batch):
    pixel_values = torch.stack([item["pixel_values"] for item in batch])
    labels = torch.tensor([item["label"] for item in batch])
    return pixel_values, labels

# Using the full train and validation splits from your HF repo!
train_loader = DataLoader(dataset["train"], batch_size=128, shuffle=False, num_workers=2, collate_fn=collate_fn)
test_loader = DataLoader(dataset["validation"], batch_size=128, shuffle=False, num_workers=2, collate_fn=collate_fn)

# ================= RESNET34 FEATURE EXTRACTOR =================
print("\nLoading ResNet34 backbone...")
backbone = resnet34(weights=ResNet34_Weights.DEFAULT)

# TRICK: Replace the final classification layer with an Identity layer 
# so the model outputs the raw 512-dim features directly!
backbone.fc = nn.Identity()
backbone = backbone.to(device)

# Freeze all backbone parameters
for param in backbone.parameters():
    param.requires_grad = False

backbone.eval()

@torch.no_grad()
def extract(loader, name):
    feats, labels = [], []
    for x, y in tqdm(loader, desc=f"Extract {name}"):
        x = x.to(device)

        # Look how much cleaner this is now!
        features = backbone(x) # (B, 512)

        feats.append(features.cpu().numpy())
        labels.append(y.numpy())

    return np.vstack(feats), np.hstack(labels)

print("\nExtracting features...")
X_train, y_train = extract(train_loader, "TRAIN")
X_test, y_test = extract(test_loader, "TEST")

print("Raw feature dim:", X_train.shape[1])

# ================= PCA =================
print("\nApplying PCA...")
pca = PCA(n_components=256, random_state=42)
X_train = pca.fit_transform(X_train)
X_test = pca.transform(X_test)

print("Reduced dim:", X_train.shape[1])

# ================= CLASSIFIER =================
print("\nTraining Logistic Regression...")
clf = LogisticRegression(max_iter=2000, n_jobs=-1, C=1.0)
clf.fit(X_train, y_train)

# ================= TRAIN =================
train_pred = clf.predict(X_train)
train_acc = accuracy_score(y_train, train_pred)
train_f1 = f1_score(y_train, train_pred, average="macro")

# ================= TEST =================
test_pred = clf.predict(X_test)
test_acc = accuracy_score(y_test, test_pred)
test_f1 = f1_score(y_test, test_pred, average="macro")

# ================= RESULT =================
print("\n==============================")
print("TRAIN RESULT")
print(f"Accuracy : {train_acc:.4f}")
print(f"F1 Score : {train_f1:.4f}")

print("\nTEST RESULT")
print(f"Accuracy : {test_acc:.4f}")
print(f"F1 Score : {test_f1:.4f}")
print("==============================")

Using device: cuda

Downloading/Loading dataset from Hugging Face...

Loading ResNet34 backbone...

Extracting features...


Extract TEST: 100%|██████████| 17/17 [00:04<00:00,  3.70it/s]


Raw feature dim: 512

Applying PCA...
Reduced dim: 256

Training Logistic Regression...

TRAIN RESULT
Accuracy : 0.8144
F1 Score : 0.7837

TEST RESULT
Accuracy : 0.8081
F1 Score : 0.7817


In [16]:
# ============================================================
# PLACES365 CLASSIFICATION: EFFICIENTNETV2-S (224x224)
# PIPELINE: HF DATASET -> EFFNET -> PCA (256) -> LOGISTIC REG
# ============================================================

import os
import random
import numpy as np
from tqdm import tqdm
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.models import efficientnet_v2_s, EfficientNet_V2_S_Weights

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.decomposition import PCA
from datasets import load_dataset

# 1. Environment & Reproducibility
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🚀 Using device: {device}")

# ================= 2. DATASET SETUP (224x224) =================
print("\n📦 Loading dataset from Hugging Face...")
dataset = load_dataset("Punnarunwuwu/industry-verification-seml")

# Standard 224x224 transformation
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

def apply_transforms(batch):
    pixel_values = []
    for img in batch["image"]:
        try:
            # Standardizing to RGB and 224x224
            pixel_values.append(transform(img.convert("RGB")))
        except Exception:
            # Fallback tensor MUST be 224x224
            pixel_values.append(torch.zeros(3, 224, 224))
            
    batch["pixel_values"] = pixel_values
    return batch

# Set lazy transform for Hugging Face dataset
dataset.set_transform(apply_transforms)

def collate_fn(batch):
    pixel_values = torch.stack([item["pixel_values"] for item in batch])
    labels = torch.tensor([item["label"] for item in batch])
    return pixel_values, labels

# DataLoaders
train_loader = DataLoader(dataset["train"], batch_size=128, shuffle=False, num_workers=2, collate_fn=collate_fn)
test_loader = DataLoader(dataset["validation"], batch_size=128, shuffle=False, num_workers=2, collate_fn=collate_fn)

# ================= 3. FEATURE EXTRACTOR =================
print("\n🧠 Loading EfficientNetV2-S backbone...")
backbone = efficientnet_v2_s(weights=EfficientNet_V2_S_Weights.DEFAULT).to(device)

# Swap classifier for Identity to get the 1280-dim embedding
backbone.classifier = nn.Identity()

# Freeze weights (Extracting features only)
for param in backbone.parameters():
    param.requires_grad = False

backbone.eval()

@torch.no_grad()
def extract_features(loader, name):
    feats, labels = [], []
    for x, y in tqdm(loader, desc=f"Extracting {name}"):
        x = x.to(device)
        # EfficientNet outputs (Batch, 1280) regardless of input resolution
        output = backbone(x)
        feats.append(output.cpu().numpy())
        labels.append(y.numpy())
    return np.vstack(feats), np.hstack(labels)

print("\n⚙️ Running Feature Extraction...")
X_train, y_train = extract_features(train_loader, "TRAIN")
X_test, y_test = extract_features(test_loader, "TEST")

print(f"✅ Raw Feature Dimension: {X_train.shape[1]}")

# ================= 4. PCA (256 COMPONENTS) =================
print("\n📉 Applying PCA (Reducing to 256 dimensions)...")
pca = PCA(n_components=256, random_state=42)
X_train_pca = pca.fit_transform(X_train)
X_test_pca = pca.transform(X_test)

print(f"✅ Reduced Feature Dimension: {X_train_pca.shape[1]}")

# ================= 5. CLASSIFIER =================
print("\n⚖️ Training Logistic Regression Classifier...")
clf = LogisticRegression(max_iter=2000, n_jobs=-1, C=1.0)
clf.fit(X_train_pca, y_train)

# ================= 6. RESULTS =================
def get_metrics(X, y):
    preds = clf.predict(X)
    acc = accuracy_score(y, preds)
    f1 = f1_score(y, preds, average="macro")
    return acc, f1

train_acc, train_f1 = get_metrics(X_train_pca, y_train)
test_acc, test_f1 = get_metrics(X_test_pca, y_test)

print("\n" + "="*40)
print("🏁 FINAL MODEL PERFORMANCE (224x224)")
print("="*40)
print(f"TRAIN | Accuracy: {train_acc:.4f} | F1: {train_f1:.4f}")
print(f"TEST  | Accuracy: {test_acc:.4f}  | F1: {test_f1:.4f}")
print("="*40)

🚀 Using device: cuda

📦 Loading dataset from Hugging Face...

🧠 Loading EfficientNetV2-S backbone...

⚙️ Running Feature Extraction...


Extracting TEST: 100%|██████████| 17/17 [00:08<00:00,  2.10it/s]


✅ Raw Feature Dimension: 1280

📉 Applying PCA (Reducing to 256 dimensions)...
✅ Reduced Feature Dimension: 256

⚖️ Training Logistic Regression Classifier...

🏁 FINAL MODEL PERFORMANCE (224x224)
TRAIN | Accuracy: 0.8497 | F1: 0.8217
TEST  | Accuracy: 0.8495  | F1: 0.8292
